In [2]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

In [4]:
model = load_model("smile_cnn_model.keras")

print("Model loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

Model loaded successfully!
Input shape: (None, 64, 64, 3)
Output shape: (None, 1)


In [5]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

print("Face detector loaded!")

Face detector loaded!


In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Webcam could not be opened!")

else:

    print("Webcam started.")
    print("Press Q to close.")

    while True:

        ret, frame = cap.read()

        if not ret:
            print("Could not read frame.")
            break
        frame = cv2.flip(frame, 1)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray,scaleFactor=1.2,minNeighbors=5,minSize=(60, 60))
        for (x, y, w, h) in faces:
            face = frame[y:y+h, x:x+w]
            face = cv2.resize(face, (64, 64))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
            face = face.astype("float32") / 255.0
            face = np.expand_dims(face, axis=0)
            prediction = model.predict(face,verbose=0)[0][0]
            if prediction >= 0.5:
                label = "SMILE"
                confidence = prediction * 100
            else:
                label = "NON-SMILE"
                confidence = (1 - prediction) * 100
            cv2.rectangle(frame,(x, y),(x + w, y + h),(0, 255, 0),2)
            text = f"{label} {confidence:.1f}%"
            cv2.putText(frame,text,(x, y - 10),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0, 255, 0),2)
        cv2.imshow(
            "Smile Detection - Press Q to Exit",
            frame
        )
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()

    print("Webcam closed.")

Webcam started.
Press Q to close.
